# Coding Homework 07: SVD, Vector Norms, and Low-Rank Weights

This assignment connects the singular value decomposition to compression of a
learned weight matrix. You will build and interpret a truncated SVD, use vector
norms to clip unusually long rows, and add one low-rank-compression function to
your cumulative mini-transformer.

**Estimated time:** 2–3 hours
**Submit:** this notebook and `mini_transformer.py`


## Start here: how to work through this notebook

You do not need to remember an earlier programming course. Work one small step
at a time. The mathematics is the goal; the programming examples are here to
help you express it.

1. Save your own copy of the notebook. A **cell** is one block of text or code.
   Click a code cell and press **Shift+Enter** to run it. Run cells from the top
   down: later cells use names created by earlier cells.
2. Run the setup cell first. `import numpy as np` makes NumPy's array tools
   available under the short name `np`.
3. Read the explanation and run the small practice example before attempting
   the corresponding task. Practice cells are supplied; you may experiment
   with them, but restore them before the final run.
4. Edit only the requested task or response. `#` begins a comment. `pass` is a
   placeholder that does nothing; a function ending at `pass` returns `None`
   (no result). Replace it with your calculation and a `return` statement.
5. Running a cell containing `def` stores the function; it does not call it.
   After editing the definition, run that cell again, then run its check cell.
   A successful `assert` may print nothing. A skipped check is not a pass:
   if a result is still `None`, the task is unfinished.
6. If a check fails, stop there. Read the last line of the error, print the
   relevant array's `.shape`, and fix this task before running cells that need it.
   Errors below an unfinished task are expected and are not separate failures.
7. Save, restart the kernel/runtime (the Python session), and run everything
   from the beginning once your work is complete. This catches accidental use
   of a value left over from an earlier attempt.

**Reading code:** `=` stores a value; `==` compares values; `@` multiplies
matrices; `*` multiplies entries or scales an array; `**2` squares entries.
Python indices start at **0**: mathematical entry $a_{1,1}$ is `A[0, 0]`.
`A.shape` is `(number_of_rows, number_of_columns)`, `A.ndim` counts dimensions,
and `A.T` transposes a matrix. An ordinary vector has shape `(n,)`.

**If you get stuck:** record the function name, the smallest input that fails,
its shape, the expected mathematical result, and the last line of the error.
Include these when asking for help. You do not need to rewrite the whole cell.

**Plan your time:** if Python is new to you, allow extra time beyond the
headline estimate and work in several short sessions. Start with setup and
the practice example; then finish one task at a time.

More worked examples and an error-message checklist: [Python and NumPy help](`mt_utils.py` / `mini_transformer.py` in this folder).


### Which file do I edit?

Download the notebook and `mini_transformer.py` from this assignment's
`student` folder. Keep both in the same working folder. Use the supplied
completed starting model; you do not need to recover earlier submissions.

- **Notebook:** fill the named notebook functions and written responses.
- **`mini_transformer.py`:** open this plain-text Python file in an editor,
  find the named milestone function, replace its placeholder, and save.
  Keep the other supplied model functions. A `.py` file cannot see a function
  merely because you defined it in the notebook.
- **Colab:** use File → Upload notebook; in the Files panel upload the `.py`
  file. Double-click that file to edit it and save. If it is not editable in
  your interface, edit it locally and upload the replacement. Download both
  edited files before ending the session; Colab uploads are temporary.
- **Local Jupyter:** open the notebook from the folder containing both files.
  If setup reports `ModuleNotFoundError: mini_transformer`, check that folder
  and the spelling before trying to install anything.

Python remembers imported files. After saving a module edit, use the reload
cell beside the milestone (or restart and run from the top). Submit **both**
files to the course portal. Wait for the results and inspect the feedback; upload
alone does not confirm the work passed.


In [1]:
# Supplied practice: run this unchanged, then trace the result by hand.
import numpy as np

def double_and_total(values):
    values = np.asarray(values, dtype=float)  # turn a list into a numerical array
    doubled = 2 * values                     # one intermediate calculation
    total = np.sum(doubled)                   # add all entries
    return doubled, total                    # send two results back to the caller

practice_vector, practice_total = double_and_total([1, -2, 3])
print(practice_vector)        # [ 2. -4.  6.]
print(practice_vector.shape)  # (3,)
print(practice_total)         # 4.0
assert practice_total == 4.0


[ 2. -4.  6.]
(3,)
4.0


## 0. Setup and cumulative-model check

Keep `mini_transformer.py` beside this notebook. It contains the completed
HW03–HW06 attention, feed-forward, residual, and feature-projection work.
Your new model function belongs in the supplied student-owned module.

A **weight matrix** stores the coefficients of a learned linear map. A
**low-rank approximation** replaces it with a matrix using fewer independent
input/output patterns. This can reduce storage and computation, but it also
changes the model's outputs.


In [2]:
import importlib
import numpy as np
import mini_transformer as mt

mt = importlib.reload(mt)
np.set_printoptions(precision=4, suppress=True)


## 1. Read a reduced SVD

For an $m\times n$ matrix $A$, the reduced (or compact) SVD is

$$A=U\operatorname{diag}(s)V^T,$$

where $r=\min(m,n)$, $U$ has shape $m\times r$, `s` contains the nonnegative
singular values in decreasing order, and $V^T$ has shape $r\times n$. The
columns of $U$ and the rows of $V^T$ are orthonormal.

Implement `compact_svd`. NumPy already computes the decomposition; your job is
to request the reduced shapes and understand the returned objects.


### Task plan: read NumPy's three outputs

Input `A` is `(m, n)`. The shape checks are supplied. Call
`np.linalg.svd(A, full_matrices=False)`, unpack its three outputs into `U`,
`s`, and `Vt`, and return those three objects in that order. `False` requests
the reduced shapes described above. `Vt` is already transposed; do not
transpose it again for reconstruction. `np.diag(s)` makes a diagonal matrix
from a vector. Inspect the shapes in the next cell before multiplying.


In [3]:
def compact_svd(A):
    """Return the reduced SVD A = U @ diag(s) @ Vt."""
    A = np.asarray(A, dtype=float)
    if A.ndim != 2 or 0 in A.shape:
        raise ValueError("A must be a nonempty matrix")
    U, s, Vt = np.linalg.svd(A, full_matrices=False)
    return U, s, Vt


In [4]:
A = np.array([[3.0, 0.0], [0.0, 2.0], [0.0, 0.0]])
U, singular_values, Vt = compact_svd(A)
assert U.shape == (3, 2)
assert singular_values.shape == (2,)
assert Vt.shape == (2, 2)
np.testing.assert_allclose(U @ np.diag(singular_values) @ Vt, A, atol=1e-10)
np.testing.assert_allclose(singular_values, [3.0, 2.0], atol=1e-10)


## 2. Keep only the strongest singular directions

The rank-$k$ truncated SVD

$$A_k=U_k\operatorname{diag}(s_1,\ldots,s_k)V_k^T$$

is the closest rank-at-most-$k$ matrix to $A$ in Frobenius norm. The Frobenius
norm is the square root of the sum of the squared entries. Implement the
truncation and support $k=0$, which should return the zero matrix.


### Task plan: slice, then reconstruct

1. Call your `compact_svd(A)` and unpack all three outputs.
2. Keep the first `k` columns of `U`, the first `k` entries of `s`, and the
   first `k` rows of `Vt`.
3. Reconstruct using the displayed three-factor formula. You may use
   `np.diag` to keep this close to the mathematics.
4. Return an array with the **same shape as `A`**. With `k=0`, the empty
   slices and matrix products give a zero array of that shape.

**Slicing practice:** for `B = np.array([[1, 2, 3], [4, 5, 6]])`,
`B[:, :2]` is `[[1, 2], [4, 5]]`, whereas `B[:1, :]` is `[[1, 2, 3]]`.
The colon means “all” when used alone. The stop index is excluded.


In [5]:
def rank_k_approximation(A, k):
    """Return the truncated-SVD approximation that keeps k singular terms."""
    A = np.asarray(A, dtype=float)
    if A.ndim != 2 or 0 in A.shape:
        raise ValueError("A must be a nonempty matrix")
    if not isinstance(k, (int, np.integer)):
        raise TypeError("k must be an integer")
    if k < 0 or k > min(A.shape):
        raise ValueError("k must satisfy 0 <= k <= min(A.shape)")
    U, s, Vt = compact_svd(A)
    U_k = U[:, :k]
    s_k = s[:k]
    Vt_k = Vt[:k, :]
    return U_k @ np.diag(s_k) @ Vt_k


In [6]:
A_rank1 = rank_k_approximation(A, 1)
np.testing.assert_allclose(A_rank1, [[3.0, 0.0], [0.0, 0.0], [0.0, 0.0]], atol=1e-10)
assert np.linalg.matrix_rank(A_rank1) == 1
np.testing.assert_allclose(rank_k_approximation(A, 0), np.zeros_like(A), atol=1e-10)


## 3. Quantify what the approximation keeps

The squared Frobenius size of a matrix is the sum of its squared singular
values. Therefore the fraction retained by $A_k$ is

$$\frac{s_1^2+\cdots+s_k^2}{s_1^2+\cdots+s_r^2}.$$

This is sometimes called **retained energy**. Here “energy” is only a compact
name for squared matrix size; it is not a physical quantity and does not by
itself measure model quality.


### Task plan: keep track of numerator and denominator

Use the singular-value vector returned by `compact_svd`. Square entries with
`**2`, then add them with `np.sum`. Compute the total first. If it is zero,
return the stated convention `1.0` before dividing. Otherwise compute the
sum for the first `k` entries and divide by the total. Return one number.
For singular values `[4., 3.]`, keeping the first retains `16/25`, not `4/7`.


In [7]:
def retained_energy(A, k):
    """Return the fraction of squared Frobenius size kept by rank k."""
    A = np.asarray(A, dtype=float)
    if A.ndim != 2 or 0 in A.shape:
        raise ValueError("A must be a nonempty matrix")
    if not isinstance(k, (int, np.integer)):
        raise TypeError("k must be an integer")
    if k < 0 or k > min(A.shape):
        raise ValueError("k must satisfy 0 <= k <= min(A.shape)")
    # (zero) energy by convention, so return 1.0 when the denominator is zero.
    _, s, _ = compact_svd(A)
    squared_s = s ** 2
    total_energy = np.sum(squared_s)
    if total_energy == 0.0:
        return 1.0
    return np.sum(squared_s[:k]) / total_energy


In [8]:
assert np.isclose(retained_energy(A, 1), 9 / 13)
assert np.isclose(retained_energy(A, 2), 1.0)
assert np.isclose(retained_energy(np.zeros((3, 2)), 0), 1.0)


## 4. Use vector norms to clip long rows

A row of a data matrix can represent one observation or one token activation.
**Norm clipping** keeps its direction but scales it down when its Euclidean
norm exceeds a chosen limit. Rows already below the limit, including zero rows,
stay unchanged. This is a bounded numerical operation, not a claim that large
vectors are inherently bad.


### Task plan: scale long rows only

Input and output both have shape `(n, d)`.

1. `np.linalg.norm(X, axis=1, keepdims=True)` measures each **row** and returns
   shape `(n, 1)`. For rows `[3, 4]` and `[0, 2]`, these lengths are `5` and `2`.
2. Start an array of scaling factors, all equal to one, with that same shape.
3. Identify rows whose lengths exceed `max_norm`. On those rows only, use
   `max_norm / row_length` as the scaling factor. This avoids division by zero
   for zero rows. Boolean indexing can select those entries.
4. Multiply `X` by these factors and return a new array. NumPy repeats each
   `(n, 1)` factor across its row; this is called **broadcasting**.

For threshold `3`, a row of length `5` gets factor `3/5`; a row of length `2`
gets factor `1`. Do not normalize every row to the threshold.


In [9]:
def clip_rows(X, max_norm):
    """Scale each row only when its Euclidean norm exceeds max_norm."""
    X = np.asarray(X, dtype=float)
    if X.ndim != 2:
        raise ValueError("X must be a matrix whose rows are vectors")
    if not np.isscalar(max_norm) or not np.isfinite(max_norm) or max_norm <= 0:
        raise ValueError("max_norm must be a finite positive number")
    # by zero. Return a new array with the same shape as X.
    norms = np.linalg.norm(X, axis=1, keepdims=True)
    factors = np.ones_like(norms)
    factors[norms > max_norm] = max_norm / norms[norms > max_norm]
    return X * factors
    pass


In [10]:
X = np.array([[3.0, 4.0], [0.0, 0.0], [6.0, 8.0], [1.0, -2.0]])
clipped = clip_rows(X, 5.0)
np.testing.assert_allclose(clipped[:3], [[3.0, 4.0], [0.0, 0.0], [3.0, 4.0]], atol=1e-10)
np.testing.assert_allclose(clipped[3], X[3], atol=1e-10)
assert np.all(np.linalg.norm(clipped, axis=1) <= 5.0 + 1e-10)


## 5. Mini-transformer milestone: compress a weight matrix

Implement `low_rank_weight(W, k)` in `mini_transformer.py`. It should return the
same truncated-SVD matrix as `rank_k_approximation`, while keeping all model
code in the student-owned module.

The example below compresses a query-weight matrix. A **query vector** is a
transformed token row used to compute attention scores. Replacing $W_Q$ changes
those queries, so retained Frobenius energy is useful context but does not
guarantee identical attention output.


### Model-file task and supplied experiment

Open `mini_transformer.py`, find `low_rank_weight`, and keep its validation
helper. Translate the same slicing/reconstruction steps you used above,
calling NumPy's SVD inside the module. A notebook's `compact_svd` is not
automatically available in a `.py` file. Save the module; the next cell reloads
it. Run the two-weight example, then the supplied attention comparison.
Only the compression function is your new model code.


### Check the model file before the full experiment

Open `mini_transformer.py` and search for **THIS WEEK**. The numbered plan and
tiny input/output example are now beside `low_rank_weight` itself. Complete that
function, save, then run the cell below. It reloads your file and checks a
small example before you encounter the longer model setup. An unfinished
function gives a message naming what to finish; do not skip this check.


In [11]:
import importlib
import mini_transformer as mt
mt = importlib.reload(mt)
mt.check_milestone()


HW11 project checks passed. Continue with the notebook experiment.


In [12]:
mt = importlib.reload(mt)
W = np.array([[3.0, 0.0], [0.0, 1.0]])
W_rank1 = mt.low_rank_weight(W, 1)
np.testing.assert_allclose(W_rank1, [[3.0, 0.0], [0.0, 0.0]], atol=1e-10)

tokens = np.array([[1.0, 0.0], [0.0, 1.0], [1.0, 1.0]])
W_K = np.eye(2)
W_V = np.array([[1.0, 0.5], [0.0, 1.0]])
full_output = mt.attention_head(tokens, W, W_K, W_V)
compressed_output = mt.attention_head(tokens, W_rank1, W_K, W_V)

print("retained weight energy:", retained_energy(W, 1))
print("attention-output change:", np.linalg.norm(full_output - compressed_output, "fro"))


retained weight energy: 0.9
attention-output change: 0.19985183073622434


### Short interpretation

In 2–4 sentences, explain which input direction the rank-one approximation of
`W` keeps, which direction it removes, and why retaining 90% of the squared
Frobenius size does not mean the attention output changes by exactly 10%.

**Your response:**


The rank-one approximation keeps the strongest singular direction and removes the weaker one. Keeping 90% of the squared Frobenius size is an overall matrix measure, so the attention output does not have to change by exactly 10%


## 6. Submission and AI-use note

Restart the kernel, run all cells, and confirm every assertion passes. Keep this notebook alongside `mt_utils.py` / `mini_transformer.py` in the same folder when re-running.


---

to third-party sites.*
